In [ ]:
import os, sys
_here = os.path.abspath(os.getcwd())
_root = _here if os.path.isdir(os.path.join(_here, 'pipeline')) else os.path.abspath(os.path.join(_here, os.pardir))
if _root not in sys.path:
    sys.path.insert(0, _root)
os.chdir(_root)
print('project root:', _root)

project root: /home/temari/god please no diploma/restore_punct


## Run config

In [ ]:
from pipeline.env import DATA_DIR, MODEL_DIR
from pipeline.config import RunConfig

cfg = RunConfig(
    name          = "01_baseline_ce",
    architecture  = "bert",
    loss          = "ce",
    train_files   = [f"{DATA_DIR}/train_all.json"],
    val_files     = [f"{DATA_DIR}/val_internal.json"],
    replay_files  = [],
    init_from     = None,
    num_epochs    = 3,
    learning_rate = 2e-5,
    tags          = {"stage": "1-baseline-loss-sweep"},
)
print(cfg)

RunConfig(name='01_baseline_ce', architecture='bert', loss='ce', train_files=['/home/temari/god please no diploma/restore_punct/data/train_all.json'], val_files=['/home/temari/god please no diploma/restore_punct/data/val_internal.json'], replay_files=[], init_from=None, num_epochs=3, learning_rate=2e-05, benchmarks=None, seed=42, tags={'stage': '1-baseline-loss-sweep'})


## Train

In [3]:
from pipeline.training import train_run
model = train_run(cfg)

[01_baseline_ce] architecture=bert loss=ce epochs=3 lr=2e-05


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

BertForTokenClassification LOAD REPORT from: DeepPavlov/rubert-base-cased-sentence
Key                 | Status     | 
--------------------+------------+-
pooler.dense.bias   | UNEXPECTED | 
pooler.dense.weight | UNEXPECTED | 
classifier.bias     | MISSING    | 
classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss,Precision,Recall,F1,Accuracy
1,0.560880,0.127006,0.960670,0.962417,0.960896,0.962417
2,0.481054,0.117811,0.963520,0.964744,0.963700,0.964744
3,0.434357,0.116918,0.964135,0.965516,0.964435,0.965516


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model saved -> /home/temari/god please no diploma/restore_punct/models/01_baseline_ce


## Benchmark + save results

In [4]:
from pipeline.evaluation import evaluate_and_save
reports = evaluate_and_save(cfg)

for test_name, rep in reports.items():
    wa = rep.get('weighted avg', {})
    print(f"{test_name:>14s} | F1={wa.get('f1-score', 0):.4f} P={wa.get('precision', 0):.4f} R={wa.get('recall', 0):.4f}")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[01_baseline_ce] evaluating on General_Test (n=569)


Evaluating:   0%|          | 0/143 [00:00<?, ?it/s]

[01_baseline_ce] evaluating on GERA_Test (n=1259)


Evaluating:   0%|          | 0/315 [00:00<?, ?it/s]

[01_baseline_ce] evaluating on LORuGEC_Test (n=603)


Evaluating:   0%|          | 0/151 [00:00<?, ?it/s]

Updated /home/temari/god please no diploma/restore_punct/results/models_db.json (entry: 01_baseline_ce)
Wrote /home/temari/god please no diploma/restore_punct/results/01_baseline_ce.json
Wrote /home/temari/god please no diploma/restore_punct/results/01_baseline_ce.xlsx
  General_Test | F1=0.9466 P=0.9450 R=0.9502
     GERA_Test | F1=0.9563 P=0.9574 R=0.9579
  LORuGEC_Test | F1=0.9654 P=0.9665 R=0.9654


## Example run

In [5]:
from pipeline.inference import load_for_inference, restore_punctuation

m, tok = load_for_inference(cfg)
for t in [
    "Мама мыла раму а папа читал газету",
    "Однако мы решили не идти в кино потому что пошел дождь",
    "Он сказал Привет как дела",
    "Я говорю ей Что думаешь А она мне Да ничего отстань уже",
    "После многих страданий А А Пушкин всё-таки написал свои стишки",
]:
    print(f"Input:    {t}")
    print(f"Restored: {restore_punctuation(m, tok, t)}\n")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Input:    Мама мыла раму а папа читал газету
Restored: Мама мыла раму, а папа читал газету.

Input:    Однако мы решили не идти в кино потому что пошел дождь
Restored: Однако мы решили не идти в кино, потому что пошел дождь.

Input:    Он сказал Привет как дела
Restored: Он сказал: " Привет, как дела".

Input:    Я говорю ей Что думаешь А она мне Да ничего отстань уже
Restored: Я говорю ей: " Что думаешь? А она мне! Да ничего отстань уже!

Input:    После многих страданий А А Пушкин всё-таки написал свои стишки
Restored: После многих страданий А. А. Пушкин всё-таки написал свои стишки.



In [6]:
from pipeline.aggregate import rebuild_master_excel
rebuild_master_excel()

Rebuilt master table -> /home/temari/god please no diploma/restore_punct/results/master_summary.xlsx
  BERT runs   : 1
  Yandex runs : 0


'/home/temari/god please no diploma/restore_punct/results/master_summary.xlsx'